In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
##############PREDEFINED FUNTIONS(JUST RUN IT)#############

import math as _m
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import numpy as np
from PIL import Image
from torchvision import transforms
from diffusers import AutoencoderKL, DDPMScheduler
from accelerate import Accelerator
import timm
import os, math, random, logging
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.transforms import functional as TF
import torchvision.utils as vutils
import matplotlib.pyplot as plt

from accelerate import Accelerator
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL

In [3]:
##############PREDEFINED FUNTIONS(JUST RUN IT)#############

import math as _m
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import numpy as np
from PIL import Image
from torchvision import transforms
from diffusers import AutoencoderKL, DDPMScheduler
from accelerate import Accelerator
import timm
import os, math, random, logging
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.transforms import functional as TF
import torchvision.utils as vutils
import matplotlib.pyplot as plt

from accelerate import Accelerator
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL

class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1
    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)
    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, num_patches, dim]
    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock2d(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=None):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.act1  = nn.SiLU()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.act2  = nn.SiLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip  = nn.Conv2d(in_channels, out_channels, 1) if in_channels!=out_channels else nn.Identity()
        if time_dim is not None:
            self.to_time = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_channels))
        else:
            self.to_time = None

    def forward(self, x, t_emb=None):
        h = self.conv1(self.act1(self.norm1(x)))
        if self.to_time is not None and t_emb is not None:
            # FiLM-like add
            h = h + self.to_time(t_emb)[:, :, None, None]
        h = self.conv2(self.act2(self.norm2(h)))
        return h + self.skip(x)

# -----------------------------
# Cond Encoder (your original for stage-1 tokens)
# -----------------------------
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=736, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64), # [B, 64, 64, 64]
            nn.AvgPool2d(2), # [B, 64, 32, 32]
            ResidualBlock(64, 128),
            nn.AvgPool2d(2), # [B, 128, 16, 16]
            ResidualBlock(128, 256),
            nn.AvgPool2d(2), # [B, 256, 8, 8]
            nn.Conv2d(256, out_channels, kernel_size=1) # [B, 736, 8, 8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),  # [B, 736, 64]
            Transpose(-1, -2),   # [B, 64, 736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)          # [B, 736, 8, 8]
        tokens = self.proj(feat)        # [B, 64, 736]
        tokens = self.pos_embed(tokens) # [B, 64, 736]
        tokens = self.norm(tokens)
        return tokens

def _g(n, c):  # GroupNorm
    return _m.gcd(n, c) or 1
##First part of encoder trainer
class SinCosPos2D(nn.Module):
    def __init__(self, H=64, W=64):
        super().__init__()
        yy, xx = torch.meshgrid(
            torch.linspace(0, 1, H), torch.linspace(0, 1, W), indexing='ij'
        )
        pe = torch.stack([
            torch.sin(2*_m.pi*xx), torch.cos(2*_m.pi*xx),
            torch.sin(2*_m.pi*yy), torch.cos(2*_m.pi*yy)
        ], dim=0)  # [4,H,W]
        self.register_buffer("pe", pe.float(), persistent=False)

    def forward(self, B):
        return self.pe.unsqueeze(0).repeat(B, 1, 1, 1)  # [B,4,H,W]

class ResInceptionDilated(nn.Module):
    def __init__(self, ch, mid=None):
        super().__init__()
        mid = mid or ch // 2

        self.pre = nn.Sequential(
            nn.GroupNorm(_g(8, ch), ch),
            nn.SiLU()
        )
        self.reduce = nn.Conv2d(ch, mid, 1)

        self.b1 = nn.Conv2d(mid, mid, 3, padding=1, dilation=1, groups=1, bias=False)
        self.b2 = nn.Conv2d(mid, mid, 3, padding=2, dilation=2, groups=1, bias=False)
        self.b3 = nn.Conv2d(mid, mid, 3, padding=4, dilation=4, groups=1, bias=False)

        self.fuse = nn.Sequential(
            nn.GroupNorm(_g(8, mid*3), mid*3),
            nn.SiLU(),
            nn.Conv2d(mid*3, ch, 1)
        )

    def forward(self, x):
        h = self.pre(x)
        h = self.reduce(h)
        h1 = self.b1(h)
        h2 = self.b2(h)
        h3 = self.b3(h)
        h  = torch.cat([h1, h2, h3], dim=1)
        h  = self.fuse(h)
        return x + h

class UpStage(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)
        self.rb   = ResInceptionDilated(ch, mid=ch//2)
    def forward(self, x):
        x = self.up(x)
        x = self.conv(x)
        x = self.rb(x)
        return x

class LatentAdapter(nn.Module):

    def __init__(self, cz=4, cond_ch=64, width=128,
                 num_blocks_64=3, include_posenc=True):
        super().__init__()
        self.include_posenc = include_posenc

        in_ch = cz + 4

        # 64×64
        self.in_conv = nn.Sequential(
            nn.Conv2d(in_ch, width, 3, padding=1),
            nn.GroupNorm(_g(8, width), width),
            nn.SiLU()
        )
        # 64×64
        self.blocks64 = nn.Sequential(*[ResInceptionDilated(width, mid=width//2) for _ in range(num_blocks_64)])


        self.up1 = UpStage(width)   # -> 128
        self.up2 = UpStage(width)   # -> 256
        self.up3 = UpStage(width)   # -> 512

        def head():
            return nn.Sequential(
                nn.Conv2d(width, cond_ch, 1),
                nn.GroupNorm(_g(8, cond_ch), cond_ch),
                nn.SiLU()
            )
        self.out64  = head()
        self.out128 = head()
        self.out256 = head()
        self.out512 = head()

        self.posenc = SinCosPos2D(64, 64) if include_posenc else None

    def forward(self, z64):
        """
        z64:    [B,4,64,64]
        mask64: [B,1,64,64] or None
        """
        B, _, H, W = z64.shape
        feats = [z64]


        if self.include_posenc:
            feats.append(self.posenc(B).to(z64.dtype).to(z64.device))

        x = torch.cat(feats, dim=1)             # [B,in_ch,64,64]
        x = self.in_conv(x)                     # [B,width,64,64]
        x = self.blocks64(x)

        f64  = self.out64(x)                    # [B,cond_ch,64,64]
        x128 = self.up1(x)
        f128 = self.out128(x128)                # [B,cond_ch,128,128]
        x256 = self.up2(x128)
        f256 = self.out256(x256)                # [B,cond_ch,256,256]
        x512 = self.up3(x256)
        f512 = self.out512(x512)                # [B,cond_ch,512,512]

        return {"s64": f64, "s128": f128, "s256": f256, "s512": f512}

class PrecomputedCascadeDataset(Dataset):
    def __init__(self, index_jsonl):
        self.items = []
        with open(index_jsonl, "r") as f:
            for line in f:
                path = json.loads(line)["pt"]
                if os.path.exists(path):
                    self.items.append(path)
                else:
                    print(f"[WARN] Missing file, skipped: {path}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        pack = torch.load(self.items[i], map_location="cpu")
        return (
            pack["masked_img"].float(),     # [-1,1]
            pack["target_img"].float(),     # [-1,1]
            torch.tensor(pack["bbox"], dtype=torch.float32),
            pack["z_cond"].to(torch.float16)  # [4,64,64]
        )


def timestep_embedding(timesteps, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(0, half, dtype=torch.float32, device=timesteps.device) / half
    )
    args = timesteps.float()[:, None] * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=1)
    if dim % 2:
        emb = torch.cat([emb, emb[:, :1]], dim=1)
    return emb


class Down(nn.Module):
    def __init__(self, in_ch, out_ch, tdim, cond_ch=0):
        super().__init__()
        self.block1 = ResidualBlock2d(in_ch + cond_ch, out_ch, time_dim=tdim)
        self.block2 = ResidualBlock2d(out_ch, out_ch, time_dim=tdim)
        self.down = nn.AvgPool2d(2)

    def forward(self, x, t_emb, cond=None):
        if cond is not None:
            x = torch.cat([x, cond], dim=1)
        x = self.block1(x, t_emb)
        x = self.block2(x, t_emb)
        x_down = self.down(x)
        return x, x_down  # return skip, next


class Up(nn.Module):
    def __init__(self, in_ch, out_ch, tdim, cond_ch=0):
        super().__init__()
        self.block1 = ResidualBlock2d(in_ch + cond_ch, out_ch, time_dim=tdim)
        self.block2 = ResidualBlock2d(out_ch, out_ch, time_dim=tdim)

    def forward(self, x, skip, t_emb, cond=None):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        if cond is not None:
            x = torch.cat([x, cond], dim=1)
        x = self.block1(x, t_emb)
        x = self.block2(x, t_emb)
        return x


from timm.models.vision_transformer import Attention as ViTAttention
import torch.nn as nn
import math as _m


def _g(n, c):  # GroupNorm
    return _m.gcd(n, c) or 1


class TimmAttn2D(nn.Module):
    def __init__(self, dim, num_heads=4, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.norm = nn.GroupNorm(_g(8, dim), dim)
        self.attn = ViTAttention(
            dim=dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=proj_drop,
        )

    def forward(self, x):  # x: [B,C,H,W]
        B, C, H, W = x.shape
        x = self.norm(x)
        x = x.flatten(2).transpose(1, 2)   # [B, HW, C]
        x = self.attn(x)                   # [B, HW, C]
        x = x.transpose(1, 2).reshape(B, C, H, W)
        return x
##Begins diffusion decoder

class UNet512(nn.Module):
    def __init__(self, base_ch=128, cond_ch=64, time_dim=256):
        super().__init__()
        ch1, ch2, ch3 = base_ch, base_ch * 2, base_ch * 4

        self.time_mlp = nn.Sequential(
            nn.Linear(320, time_dim), nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        # condition
        self.in_conv = nn.Conv2d(3 + cond_ch, ch1, 3, padding=1)

        # 512->256->128->64
        self.down1 = Down(ch1, ch1, tdim=time_dim, cond_ch=0)           # 512->256
        self.down2 = Down(ch1, ch2, tdim=time_dim, cond_ch=cond_ch)     # 256->128
        self.down3 = Down(ch2, ch3, tdim=time_dim, cond_ch=cond_ch)     # 128->64

        # 64×64
        self.attn64 = TimmAttn2D(dim=ch3, num_heads=4)

        # 64×64
        self.mid1 = ResidualBlock2d(ch3 + cond_ch, ch3, time_dim=time_dim)
        self.mid2 = ResidualBlock2d(ch3, ch3, time_dim=time_dim)

        # 64->128->256->512
        self.up3 = Up(ch3 + ch3, ch2, tdim=time_dim, cond_ch=cond_ch)
        self.up2 = Up(ch2 + ch2, ch1, tdim=time_dim, cond_ch=cond_ch)
        self.up1 = Up(ch1 + ch1, ch1, tdim=time_dim, cond_ch=0)

        self.out_norm = nn.GroupNorm(8, ch1)
        self.out = nn.Conv2d(ch1, 3, 3, padding=1)

    def forward(self, x, timesteps, cond_feats):
        # t embedding
        t_emb = self.time_mlp(timestep_embedding(timesteps, 320))

        x = torch.cat([x, cond_feats["s512"]], dim=1)
        x = self.in_conv(x)

        # Down: 512->256->128->64
        skip1, x = self.down1(x, t_emb, cond=None)
        skip2, x = self.down2(x, t_emb, cond=cond_feats["s256"])
        skip3, x = self.down3(x, t_emb, cond=cond_feats["s128"])

        # 64×64
        x = self.attn64(x)
        x = torch.cat([x, cond_feats["s64"]], dim=1)
        x = self.mid1(x, t_emb)
        x = self.mid2(x, t_emb)

        # Up: 64->128->256->512
        x = self.up3(x, skip3, t_emb, cond=cond_feats["s128"])
        x = self.up2(x, skip2, t_emb, cond=cond_feats["s256"])
        x = self.up1(x, skip1, t_emb, cond=None)

        x = self.out(self.out_norm(x).clamp(-6, 6))
        x = torch.tanh(x)
        return x


from torchvision.models import vgg16
from torchvision.models.feature_extraction import create_feature_extractor
from diffusers import DDPMScheduler

class Cascade512Trainer:
    def __init__(self, train_index, val_index, bs=4, lr=1e-5):
        self.accelerator = Accelerator(mixed_precision="fp16")
        self.loss_history, self.val_loss_history = [], []

        self.train_loader = DataLoader(
            PrecomputedCascadeDataset(train_index),
            batch_size=bs, shuffle=True, num_workers=4, pin_memory=True
        )
        self.val_loader = DataLoader(
            PrecomputedCascadeDataset(val_index),
            batch_size=2, shuffle=False, num_workers=2, pin_memory=True
        )

        self.adapter = LatentAdapter(cz=4, cond_ch=96)
        self.unet512 = UNet512(base_ch=192, cond_ch=96, time_dim=256)

        self.optimizer = torch.optim.AdamW(
            list(self.adapter.parameters()) + list(self.unet512.parameters()),
            lr=lr, betas=(0.9, 0.999), weight_decay=1e-5
        )

        self.train_scheduler = DDPMScheduler.from_pretrained(
            "runwayml/stable-diffusion-v1-5", subfolder="scheduler"
        )
        self.train_scheduler.config.prediction_type = "sample"

        self.infer_scheduler = DDPMScheduler.from_pretrained(
            "runwayml/stable-diffusion-v1-5", subfolder="scheduler"
        )
        self.infer_scheduler.config.prediction_type = "sample"

        comps = [self.adapter, self.unet512, self.train_loader, self.val_loader, self.optimizer]
        self.adapter, self.unet512, self.train_loader, self.val_loader, self.optimizer = self.accelerator.prepare(*comps)

        self.vis_dir = "drive/MyDrive/vis_cascade"
        os.makedirs(self.vis_dir, exist_ok=True)

        vgg = vgg16(pretrained=True).features.eval()
        self.vgg = create_feature_extractor(vgg, return_nodes={"16": "feat"}).requires_grad_(False)

        comps = [self.adapter, self.unet512, self.train_loader, self.val_loader, self.optimizer, self.vgg]
        self.adapter, self.unet512, self.train_loader, self.val_loader, self.optimizer, self.vgg = self.accelerator.prepare(*comps)

    def _pixel_mask_from_bbox(self, bbox, img_shape):
        B, _, H, W = img_shape
        masks = torch.zeros(B, 1, H, W, device=self.accelerator.device)
        for i in range(B):
            x1, y1, x2, y2 = bbox[i]
            x1i, x2i = int(x1.item() * W), int(x2.item() * W)
            y1i, y2i = int(y1.item() * H), int(y2.item() * H)
            masks[i, :, y1i:y2i, x1i:x2i] = 1.0
        return masks

    def _step(self, batch, train=True):
        masked_imgs, target_imgs, bbox, z_cond = batch
        z_cond = z_cond.to(self.accelerator.device, dtype=torch.float16)
        cond_feats = self.adapter(z_cond)
        noise = torch.randn_like(target_imgs)
        timesteps = torch.randint(
            0, self.train_scheduler.config.num_train_timesteps,
            (target_imgs.shape[0],), device=target_imgs.device
        ).long()
        x_noisy = self.train_scheduler.add_noise(target_imgs, noise, timesteps)
        pixel_mask = self._pixel_mask_from_bbox(bbox, target_imgs.shape)
        x0_pred = self.unet512(x_noisy, timesteps, cond_feats)
        loss_mse = F.mse_loss(x0_pred, target_imgs)

        with torch.no_grad():
            target_01 = (target_imgs.clamp(-1, 1) + 1) / 2
        pred_01 = (x0_pred.clamp(-1, 1) + 1) / 2
        vgg_feat_pred = self.vgg(pred_01)["feat"]
        vgg_feat_target = self.vgg(target_01)["feat"]
        loss_vgg = F.l1_loss(vgg_feat_pred, vgg_feat_target)

        loss = 0.9 * loss_mse + 0.1 * loss_vgg

        if train:
            self.accelerator.backward(loss)
        return loss

    @torch.no_grad()
    def validate(self):
        self.unet512.eval(); self.adapter.eval()
        total, n = 0.0, 0
        for batch in tqdm(self.val_loader, desc="Validating"):
            loss = self._step(batch, train=False)
            total += loss.item(); n += 1
        return total / max(1, n)

    @torch.no_grad()
    def visualize_epoch(self, epoch_idx: int, max_batches: int = 1, steps_stage2: int = 50):
        self.unet512.eval(); self.adapter.eval()
        saved = 0; grids = []
        self.infer_scheduler.set_timesteps(steps_stage2, device=self.accelerator.device)
        for batch in self.val_loader:
            masked_imgs, target_imgs, bbox, z_cond = batch
            z_cond = z_cond.to(self.accelerator.device, dtype=torch.float16)
            cond_feats = self.adapter(z_cond)

            B, _, H, W = masked_imgs.shape
            x = torch.randn(B, 3, H, W, device=self.accelerator.device) * self.infer_scheduler.init_noise_sigma
            for t in self.infer_scheduler.timesteps:
                t_batch = torch.full((B,), int(t), device=self.accelerator.device, dtype=torch.long)
                x0_pred = self.unet512(x, t_batch, cond_feats)
                x = self.infer_scheduler.step(x0_pred, t, x).prev_sample  # DDIM deterministic

            pred = (x.clamp(-1, 1) + 1) / 2
            masked_vis = (masked_imgs.clamp(-1, 1) + 1) / 2
            target_vis = (target_imgs.clamp(-1, 1) + 1) / 2
            triplet = torch.cat([masked_vis, target_vis, pred], dim=0)
            grid = vutils.make_grid(triplet, nrow=B, padding=2)
            grids.append(grid); saved += 1
            if saved >= max_batches: break
        if grids:
            big = torch.cat(grids, dim=1) if len(grids) > 1 else grids[0]
            out_path = os.path.join(self.vis_dir, f"epoch_{epoch_idx:03d}.png")
            vutils.save_image(big, out_path)
            self.accelerator.print(f"[Visualize] Saved {out_path}")

    @torch.no_grad()
    def eval_composition_batch(self, ae_model, index, steps_stage2=50, save_dir="comp_eval"):
        self.unet512.eval(); self.adapter.eval()
        os.makedirs(save_dir, exist_ok=True)

        self.infer_scheduler.set_timesteps(steps_stage2, device=self.accelerator.device)

        batch = next(iter(self.val_loader))
        masked_imgs, target_imgs, bbox, z_cond = batch
        B = masked_imgs.size(0)

        z_cond = z_cond.to(self.accelerator.device, dtype=torch.float16)
        cond_feats = self.adapter(z_cond)

        x = torch.randn(B, 3, masked_imgs.shape[2], masked_imgs.shape[3],
                        device=self.accelerator.device) * self.infer_scheduler.init_noise_sigma
        for t in self.infer_scheduler.timesteps:
            t_batch = torch.full((B,), int(t), device=self.accelerator.device, dtype=torch.long)
            x0_pred = self.unet512(x, t_batch, cond_feats)
            x = self.infer_scheduler.step(x0_pred, t, x).prev_sample

        pred_imgs = (x.clamp(-1, 1) + 1) / 2
        masked_vis = (masked_imgs.clamp(-1, 1) + 1) / 2
        target_vis = (target_imgs.clamp(-1, 1) + 1) / 2

        # composition
        fr_recon, fr_pred, fr_orig = [], [], []
        for i in range(B):
            type_recon = infer_cell_map(masked_vis[i], ae_model)
            type_pred = infer_cell_map(pred_imgs[i], ae_model)
            type_orig = infer_cell_map(target_vis[i], ae_model)

            dist_recon = compute_type_distribution(type_recon.squeeze(0).cpu().numpy(), num_types=25)
            dist_pred = compute_type_distribution(type_pred.squeeze(0).cpu().numpy(), num_types=25)
            dist_orig = compute_type_distribution(type_orig.squeeze(0).cpu().numpy(), num_types=25)

            fr_recon.append(dist_recon)
            fr_pred.append(dist_pred)
            fr_orig.append(dist_orig)

        for i in range(B):
            fig, axes = plt.subplots(3, 3, figsize=(12, 8))
            def show_img(ax, img, title):
                ax.imshow(img.permute(1, 2, 0).cpu().numpy())
                ax.set_title(title)
                ax.axis("off")

            show_img(axes[0, 0], masked_vis[i], "Recon (cond)")
            show_img(axes[0, 1], target_vis[i], "Orig (GT)")
            show_img(axes[0, 2], pred_imgs[i], "Pred")

            xs = np.arange(25)
            bar_width = 0.8
            for ax, frac, title in zip(
                axes[1],
                [fr_recon[i], fr_orig[i], fr_pred[i]],
                ["Recon comp", "Orig comp", "Pred comp"]
            ):
                ax.bar(xs, frac, width=bar_width, color="skyblue")
                ax.set_title(title)
                ax.set_ylim(0, 1)
                ax.set_xticks(xs)
                ax.set_xticklabels([f"T{i+1}" for i in xs], rotation=90, fontsize=6)
                ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
                ax.set_yticklabels(["0%", "20%", "40%", "60%", "80%", "100%"])
                ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.3)

            for ax, img, title in zip(
                axes[2],
                [masked_vis[i], target_vis[i], pred_imgs[i]],
                ["Recon RGB Hist", "Orig RGB Hist", "Pred RGB Hist"]
            ):
                img_np = img.cpu().numpy()
                colors = ["r", "g", "b"]
                for c in range(3):
                    hist, bins = np.histogram(img_np[c], bins=50, range=(0, 1))
                    ax.plot(bins[:-1], hist, color=colors[c], alpha=0.7, label=colors[c].upper())
                ax.set_title(title)
                ax.set_xlim(0, 1)
                ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
                ax.set_yticks([])
                ax.legend()

            plt.tight_layout()
            out_path = os.path.join(save_dir, f"vis_comp_img{i}_{index}.png")
            plt.savefig(out_path, dpi=150)
            plt.close()
            self.accelerator.print(f"[CompEval] Saved {out_path}")

    def train(self, epochs=30, patience=5, vis_steps_stage2=50):
        device = self.accelerator.device
        ae = Autoencoder().to(device).eval()
        ae.load_state_dict(torch.load("drive/MyDrive/newae2.pth", map_location=device))
        best = float("inf"); bad = 0
        for ep in range(1, epochs + 1):
            self.unet512.train(); self.adapter.train()
            prog = tqdm(self.train_loader, desc=f"Epoch {ep} [Train]")
            losses = []
            for batch in prog:
                with self.accelerator.accumulate(self.unet512):
                    loss = self._step(batch, train=True)
                    if self.accelerator.sync_gradients:
                        self.accelerator.clip_grad_norm_(self.unet512.parameters(), 1.0)
                    self.optimizer.step(); self.optimizer.zero_grad()
                losses.append(loss.item()); prog.set_postfix(loss=np.mean(losses))

            tr = float(np.mean(losses)); self.loss_history.append(tr)
            va = self.validate(); self.val_loss_history.append(va)
            self.accelerator.print(f"[Epoch {ep}] train={tr:.4f} val={va:.4f}")
            self.visualize_epoch(epoch_idx=ep, max_batches=4, steps_stage2=vis_steps_stage2)
            self.eval_composition_batch(ae, ep, steps_stage2=vis_steps_stage2,
                                        save_dir="drive/MyDrive/checkpoint_cascade512_best1")

            if va < 1:
                best = va; bad = 0
                self.accelerator.wait_for_everyone()
                self.accelerator.save_state("drive/MyDrive/checkpoint_cascade512_best1")
                self.accelerator.print(f"  >> Saved best at val {best:.4f}")
            else:
                bad += 1; self.accelerator.print(f"  >> No improve ({bad}/{patience})")
                if bad >= patience:
                    self.accelerator.print("Early stop."); break

In [ ]:
cell_type_names = [
    "B", "CD4+ T cell", "CD57+ Enterocyte", "CD66+ Enterocyte", "CD7+ Immune",
    "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte",
    "Goblet", "ICC", "Lymphatic", "M1 Macrophage", "M2 Macrophage",
    "MUC1+ Enterocyte", "NK", "Nerve", "Neuroendocrine", "Neutrophil",
    "Paneth", "Plasma", "Smooth muscle", "Stroma", "TA"
]


type_color_dict = {
    1: [134.9, 124.1, 8.7],
    2: [62.3, 216.0, 132.0],
    3: [130.9, 46.4, 182.5],
    4: [92.8, 63.6, 202.5],
    5: [140.1, 239.7, 189.0],
    6: [200.0, 183.1, 168.9],
    7: [108.3, 16.1, 142.3],
    8: [132.7, 217.1, 223.4],
    9: [170.6, 96.0, 98.0],
    10: [68.4, 101.1, 101.3],
    11: [4.6, 146.9, 145.6],
    12: [177.1, 1.5, 112.2],
    13: [202.6, 136.4, 12.3],
    14: [95.1, 244.3, 209.6],
    15: [32.5, 214.2, 217.8],
    16: [176.2, 78.6, 193.2],
    17: [185.9, 206.0, 77.8],
    18: [212.0, 39.3, 35.0],
    19: [117.1, 190.4, 53.4],
    20: [101.2, 145.2, 246.9],
    21: [13.7, 179.1, 133.2],
    22: [247.4, 125.5, 111.5],
    23: [122.9, 151.0, 150.0],
    24: [118.0, 45.6, 47.7],
    25: [31.1, 114.7, 222.5]
}

In [5]:
class Autoencoder(nn.Module):
    def __init__(self, in_dim=25, bottleneck_dim=3, hidden_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim)), nn.ReLU(), nn.LayerNorm(int(hidden_dim)), nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), hidden_dim), nn.ReLU(), nn.LayerNorm(int(hidden_dim)), nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), in_dim),
        )
    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)

_MIN_VALS = torch.tensor([-69.761505, -75.65188, -77.16103], dtype=torch.float32)
_MAX_VALS = torch.tensor([ 88.969406,  65.244896, 67.13518 ], dtype=torch.float32)
_RANGE    = _MAX_VALS - _MIN_VALS
THRESH_WHITE_01 = 250.0/255.0

@torch.no_grad()
def fractions_from_batch_rgb(imgs_m11, ae_model):
    device = next(ae_model.parameters()).device
    B, _, H, W = imgs_m11.shape
    x01 = imgs_m11
    mv = _MIN_VALS.to(device); rg = _RANGE.to(device); thr = THRESH_WHITE_01

    fracs = []; pixs = []
    for i in range(B):
        rgb = x01[i].to(device)                        # [3,H,W]
        flat = rgb.permute(1,2,0).reshape(-1,3)
        white = ((flat > thr)).all(dim=1)
        valid = flat[~white]
        if valid.numel() == 0:
            fracs.append(torch.zeros(25, device="cpu")); pixs.append(0); continue
        z = valid * rg + mv
        logits = ae_model.decoder(z)                   # [N,25]
        pred = torch.argmax(logits, dim=1)             # 0..24
        cnt  = torch.bincount(pred, minlength=25).float().cpu()
        fracs.append(cnt / cnt.sum().clamp_min(1))
        pixs.append(int(cnt.sum().item()))
    return torch.stack(fracs, 0), np.array(pixs, dtype=np.int64)

In [4]:
import timm
import os, math, random, logging
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.transforms import functional as TF
import torchvision.utils as vutils
import matplotlib.pyplot as plt

from accelerate import Accelerator
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL

class Autoencoder_m(nn.Module):
    def __init__(self, in_dim=34, bottleneck_dim=3, hidden_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Linear(int(hidden_dim/2), int(hidden_dim)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim)),
            nn.Linear(hidden_dim, int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Linear(int(hidden_dim/4), bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Linear(int(hidden_dim/2), hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim)),
            nn.Linear(hidden_dim, int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Linear(int(hidden_dim/4), in_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)

# z3d 缩放常量
_MIN_VALS = torch.tensor([-84.5987,  -86.36052, -74.5089], dtype=torch.float32)
_MAX_VALS = torch.tensor([87.49823, 91.5325,  71.96337], dtype=torch.float32)
_RANGE    = _MAX_VALS - _MIN_VALS
THRESH_WHITE_01 = 240.0/255.0
@torch.no_grad()
def fractions_from_batch_rgb_m(imgs_m11, ae_model):
    """
    imgs_m11: [B,3,H,W] ∈ [0,1]；组成统计时剔除“白像素”（RGB>250/255）
    返回: fractions [B,25]、valid_pixels [B]
    """
    device = next(ae_model.parameters()).device
    B, _, H, W = imgs_m11.shape
    x01 = imgs_m11
    mv = _MIN_VALS.to(device); rg = _RANGE.to(device); thr = THRESH_WHITE_01

    fracs = []; pixs = []
    for i in range(B):
        rgb = x01[i].to(device)                        # [3,H,W]
        flat = rgb.permute(1,2,0).reshape(-1,3)
        white = ((flat > thr)).all(dim=1)               # 只在统计时剔除白
        valid = flat[~white]
        if valid.numel() == 0:
            fracs.append(torch.zeros(34, device="cpu")); pixs.append(0); continue
        z = valid * rg + mv                            # 恢复 z3d
        logits = ae_model.decoder(z)                   # [N,25]
        pred = torch.argmax(logits, dim=1)             # 0..24
        cnt  = torch.bincount(pred, minlength=34).float().cpu()
        fracs.append(cnt / cnt.sum().clamp_min(1))
        pixs.append(int(cnt.sum().item()))
    return torch.stack(fracs, 0), np.array(pixs, dtype=np.int64)

def compute_type_distribution(type_map_np, num_types=34):
    counts = np.bincount(type_map_np.ravel(), minlength=num_types + 1)[1:num_types + 1].astype(float)
    total = counts.sum()
    if total <= 0:
        return np.zeros(num_types, dtype=float)
    return counts / total

def compute_jsd(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p /= p.sum(); q /= q.sum()
    return jensenshannon(p, q, base=2) ** 2

def infer_cell_map_m(img, ae_model):
    mv = torch.tensor([-84.5987,  -86.36052, -74.5089], device=img.device)
    rg = torch.tensor([87.49823, 91.5325,  71.96337], device=img.device) - mv
    H, W = img.shape[1:]
    flat = img.permute(1, 2, 0).reshape(-1, 3)
    white_mask = (flat > 0.90).all(dim=1)
    infer_input_rgb = flat[~white_mask]
    pred = torch.zeros(flat.shape[0], dtype=torch.long, device=img.device)
    if infer_input_rgb.shape[0] > 0:
        z_recovered = infer_input_rgb * rg + mv
        logits = ae_model.decoder(z_recovered)
        pred[~white_mask] = torch.argmax(logits, dim=1) + 1
    return pred.reshape(1, H, W)


In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from collections import defaultdict
import torch
import json, os, re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.spatial.distance import jensenshannon


def compute_jsd(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p /= p.sum(); q /= q.sum()
    return jensenshannon(p, q, base=2) ** 2

def load_and_recover_z3d_png(path):
    """
    Load a PNG image as RGB, interpret it as normalized z_3d embedding encoded as uint8 [0–255],
    and recover the original z_3d float values via inverse scaling.
    """
    # 1. Load RGB image as uint8 tensor
    img = Image.open(path).convert("RGB")
    arr = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes()))
    rgb = arr.view(img.size[1], img.size[0], 3).permute(2, 0, 1).clone()  # [3, H, W], uint8
    mask = (rgb > 230).all(dim=0)
    rgb[:, mask] = 255
    rgb_float = rgb.float() / 255.0  # [3, H, W], float32
    non_white_pixels = ((rgb_float != 1).any(dim=0)).sum().item()
    print(f"count for non white: {non_white_pixels}")

    return rgb_float

def infer_cell_map(latent_image, model):
    min_vals = torch.tensor([-69.761505, -75.65188,  -77.16103], device=latent_image.device)
    range_vals = torch.tensor([88.969406, 65.244896, 67.13518], device=latent_image.device) - min_vals
    H, W = latent_image.shape[1], latent_image.shape[2]
    latent_image = latent_image.to("cuda")
    range_vals = range_vals.to("cuda")
    min_vals = min_vals.to("cuda")

    print("Original RGB min/max：")
    print("  R:", latent_image[0].min().item(), latent_image[0].max().item())
    print("  G:", latent_image[1].min().item(), latent_image[1].max().item())
    print("  B:", latent_image[2].min().item(), latent_image[2].max().item())

    flat_img = latent_image.permute(1, 2, 0).reshape(-1, 3)
    white_mask = (flat_img > 0.95).all(dim=1)

    print("valid pixel：", (~white_mask).sum().item())

    infer_input_rgb = flat_img[~white_mask]
    pred = torch.zeros(flat_img.shape[0], dtype=torch.long, device="cuda")
    if infer_input_rgb.shape[0] > 0:
        z_recovered = infer_input_rgb * range_vals + min_vals
        logits = model.decoder(z_recovered)
        pred[~white_mask] = torch.argmax(logits, dim=1) + 1

    pred = pred.reshape(1, H, W)
    return pred


device = "cuda"

cell_type_names = [
    "B", "CD4+ T cell", "CD57+ Enterocyte", "CD66+ Enterocyte", "CD7+ Immune",
    "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte",
    "Goblet", "ICC", "Lymphatic", "M1 Macrophage", "M2 Macrophage",
    "MUC1+ Enterocyte", "NK", "Nerve", "Neuroendocrine", "Neutrophil",
    "Paneth", "Plasma", "Smooth muscle", "Stroma", "TA"
]

type_color_dict = {
    1: [134.9, 124.1, 8.7],
    2: [62.3, 216.0, 132.0],
    3: [130.9, 46.4, 182.5],
    4: [92.8, 63.6, 202.5],
    5: [140.1, 239.7, 189.0],
    6: [200.0, 183.1, 168.9],
    7: [108.3, 16.1, 142.3],
    8: [132.7, 217.1, 223.4],
    9: [170.6, 96.0, 98.0],
    10: [68.4, 101.1, 101.3],
    11: [4.6, 146.9, 145.6],
    12: [177.1, 1.5, 112.2],
    13: [202.6, 136.4, 12.3],
    14: [95.1, 244.3, 209.6],
    15: [32.5, 214.2, 217.8],
    16: [176.2, 78.6, 193.2],
    17: [185.9, 206.0, 77.8],
    18: [212.0, 39.3, 35.0],
    19: [117.1, 190.4, 53.4],
    20: [101.2, 145.2, 246.9],
    21: [13.7, 179.1, 133.2],
    22: [247.4, 125.5, 111.5],
    23: [122.9, 151.0, 150.0],
    24: [118.0, 45.6, 47.7],
    25: [31.1, 114.7, 222.5]
}

type_color_norm = {
    idx: (np.array(rgb) / 255.0).tolist() for idx, rgb in type_color_dict.items()
}

def compute_type_distribution(type_map_np: np.ndarray, num_types: int = 25) -> np.ndarray:
    counts = np.bincount(type_map_np.ravel(), minlength=num_types + 1)[1:num_types + 1].astype(float)
    total = counts.sum()
    if total <= 0:
        return np.zeros(num_types, dtype=float)
    return counts / total

def save_inferred_map_to_df(cell_type_map: torch.Tensor, region_id=0) -> pd.DataFrame:
    if cell_type_map.ndim == 3:
        cell_type_map = cell_type_map.squeeze(0)  # [H, W]
    cell_type_np = cell_type_map.detach().cpu().numpy()  # [H, W]
    ys, xs = np.where(cell_type_np > 0)
    types = cell_type_np[ys, xs].astype(int)
    df = pd.DataFrame({
        "Cell Type": [cell_type_names[t - 1] for t in types],
        "x": xs,
        "y": ys,
        "unique_region": region_id
    })
    return df


In [ ]:
##############MAIN FUNCTION ENTRANCE（from image to image）#############

png_path = "/content/drive/MyDrive/202509CURRENT_Diff_TRAIN_set/region_9.png" #######change this, images are in drive/MyDrive/202509CURRENT_Diff_TRAIN_set
save_dir = "/content/drive/MyDrive/20251006decoded_eval"
vae_path = "runwayml/stable-diffusion-v1-5"
ckpt_path = "/content/drive/MyDrive/checkpoint_cascade512_best1"
os.makedirs(save_dir, exist_ok=True)


# Device and accelerator
device = torch.device("cuda")
accelerator = Accelerator(mixed_precision='fp16')


# Load VAE (for encoding)
vae = AutoencoderKL.from_pretrained(vae_path, subfolder="vae", torch_dtype=torch.float16).to(device)
vae.eval()


# Load diffusion model + adapter
adapter = LatentAdapter(cz=4, cond_ch=96)
unet = UNet512(base_ch=192, cond_ch=96, time_dim=256)
scheduler = DDPMScheduler.from_pretrained(vae_path, subfolder="scheduler")
scheduler.set_timesteps(50, device=device)
scheduler.config.prediction_type = "sample"


adapter, unet = accelerator.prepare(adapter, unet)
accelerator.load_state(ckpt_path)
adapter.eval(); unet.eval()


# Preprocess input image
image = Image.open(png_path).convert("RGB").resize((512, 512))
transform = transforms.Compose([
transforms.ToTensor(),
transforms.Normalize([0.5]*3, [0.5]*3) # → [-1, 1]
])
image_tensor = transform(image).unsqueeze(0).to(device, dtype=torch.float16) # [1,3,512,512]


# Encode with VAE
with torch.no_grad():
  z = vae.encode(image_tensor).latent_dist.sample()
  torch.save(z, os.path.join(save_dir, "latent.pt"))
# Decode with diffusion
  cond_feats = adapter(z)
  B, _, H, W = 1, 3, 512, 512
  x = torch.randn(B, 3, H, W, device=device) * scheduler.init_noise_sigma
  for t in scheduler.timesteps:
    t_batch = torch.full((B,), int(t), device=device, dtype=torch.long)
    x0_pred = unet(x, t_batch, cond_feats)
    x = scheduler.step(x0_pred, t, x).prev_sample
    out_img = (x.clamp(-1, 1) + 1) / 2


# Save output image
out_pil = transforms.ToPILImage()(out_img[0].float().cpu())
out_pil.save(os.path.join(save_dir, "20251006Region9RecoveredNogeneration.png"))
print(f"Saved latent + painted image to {save_dir}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
import os
import torch
from PIL import Image
from accelerate import Accelerator
from diffusers import AutoencoderKL, DDPMScheduler
from torchvision import transforms

############## MAIN FUNCTION ENTRANCE（from image to image, batch folder）#############

# === 路径设置 ===
input_dir = "/content/drive/MyDrive/202509CURRENT_Diff_TRAIN_set"  # 输入文件夹
save_dir = "/content/drive/MyDrive/202509CURRENT_Diff_TRAIN_set_recover"
vae_path = "runwayml/stable-diffusion-v1-5"
ckpt_path = "/content/drive/MyDrive/checkpoint_cascade512_best1"
os.makedirs(save_dir, exist_ok=True)

# === 加载模型 ===
device = torch.device("cuda")
accelerator = Accelerator(mixed_precision='fp16')

# VAE
vae = AutoencoderKL.from_pretrained(vae_path, subfolder="vae", torch_dtype=torch.float16).to(device)
vae.eval()

# Diffusion模型 + Adapter
adapter = LatentAdapter(cz=4, cond_ch=96)
unet = UNet512(base_ch=192, cond_ch=96, time_dim=256)
scheduler = DDPMScheduler.from_pretrained(vae_path, subfolder="scheduler")
scheduler.set_timesteps(150, device=device)
scheduler.config.prediction_type = "sample"

adapter, unet = accelerator.prepare(adapter, unet)
accelerator.load_state(ckpt_path)
adapter.eval()
unet.eval()

# === 图像预处理 ===
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)  # → [-1, 1]
])

# === 批量处理文件夹中的所有图像 ===
valid_exts = [".png", ".jpg", ".jpeg", ".webp"]
img_files = [f for f in os.listdir(input_dir) if os.path.splitext(f)[1].lower() in valid_exts]
print(f"Found {len(img_files)} images in {input_dir}")

for fname in img_files:
    in_path = os.path.join(input_dir, fname)
    base = os.path.splitext(fname)[0]
    print(f"🟢 Processing: {fname}")

    # --- 读图 + resize ---
    image = Image.open(in_path).convert("RGB").resize((512, 512))
    image_tensor = transform(image).unsqueeze(0).to(device, dtype=torch.float16)  # [1,3,512,512]

    # --- Encode with VAE ---
    with torch.no_grad():
        z = vae.encode(image_tensor).latent_dist.sample()

        # --- Decode with diffusion ---
        cond_feats = adapter(z)
        B, _, H, W = 1, 3, 512, 512
        x = torch.randn(B, 3, H, W, device=device) * scheduler.init_noise_sigma

        for t in scheduler.timesteps:
            t_batch = torch.full((B,), int(t), device=device, dtype=torch.long)
            x0_pred = unet(x, t_batch, cond_feats)
            x = scheduler.step(x0_pred, t, x).prev_sample

        out_img = (x.clamp(-1, 1) + 1) / 2  # [0,1]
        out_pil = transforms.ToPILImage()(out_img[0].float().cpu())
        out_pil.save(os.path.join(save_dir, f"{base}_decoded.png"))

    print(f"✅ Saved latent + decoded image for {fname}")

print(f"🎯 All done! Results saved to {save_dir}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

Found 66 images in /content/drive/MyDrive/202509CURRENT_Diff_TRAIN_set
🟢 Processing: region_0.png
✅ Saved latent + decoded image for region_0.png
🟢 Processing: region_1.png
✅ Saved latent + decoded image for region_1.png
🟢 Processing: region_10.png
✅ Saved latent + decoded image for region_10.png
🟢 Processing: region_11.png
✅ Saved latent + decoded image for region_11.png
🟢 Processing: region_12.png
✅ Saved latent + decoded image for region_12.png
🟢 Processing: region_13.png
✅ Saved latent + decoded image for region_13.png
🟢 Processing: region_14.png
✅ Saved latent + decoded image for region_14.png
🟢 Processing: region_15.png
✅ Saved latent + decoded image for region_15.png
🟢 Processing: region_16.png
✅ Saved latent + decoded image for region_16.png
🟢 Processing: region_17.png
✅ Saved latent + decoded image for region_17.png
🟢 Processing: region_18.png
✅ Saved latent + decoded image for region_18.png
🟢 Processing: region_19.png
✅ Saved latent + decoded image for region_19.png
🟢 Process

In [ ]:
#######to get cell type distribution#########
def cell_map(image_tensor: torch.Tensor, ae_model: torch.nn.Module, num_types: int = 25) -> np.ndarray:
    """
    Convert an RGB image tensor to cell type composition distribution using the AE model.

    Args:
        image_tensor (torch.Tensor): Tensor of shape [3, H, W], values in [0, 1].
        ae_model (torch.nn.Module): Pretrained autoencoder model with a .decoder method.
        num_types (int): Number of target cell types (default: 25).

    Returns:
        np.ndarray: Normalized composition distribution, shape [25].
    """
    assert image_tensor.ndim == 3 and image_tensor.shape[0] == 3, "Expecting image tensor of shape [3, H, W]"
    image_tensor = image_tensor.to("cuda")

    # --- Print min/max for debug
    print("Image RGB min/max:")
    print("  R:", image_tensor[0].min().item(), image_tensor[0].max().item())
    print("  G:", image_tensor[1].min().item(), image_tensor[1].max().item())
    print("  B:", image_tensor[2].min().item(), image_tensor[2].max().item())

    # Constants to recover z from RGB
    min_vals = torch.tensor([-69.761505, -75.65188,  -77.16103], device="cuda")
    range_vals = torch.tensor([88.969406, 65.244896, 67.13518], device="cuda") - min_vals

    # Reshape
    H, W = image_tensor.shape[1], image_tensor.shape[2]
    flat_img = image_tensor.permute(1, 2, 0).reshape(-1, 3)

    # White mask (ignore white pixels)
    white_mask = (flat_img > 0.95).all(dim=1)
    print("Valid pixels:", (~white_mask).sum().item())

    # Inference
    pred = torch.zeros(flat_img.shape[0], dtype=torch.long, device="cuda")
    if (~white_mask).sum() > 0:
        infer_input_rgb = flat_img[~white_mask]
        z_recovered = infer_input_rgb * range_vals + min_vals
        logits = ae_model.decoder(z_recovered)
        pred[~white_mask] = torch.argmax(logits, dim=1) + 1  # [1,25]

    # Convert to map and compute distribution
    pred_map = pred.reshape(1, H, W)
    return pred_map


In [ ]:
#######example to get cell type distribution#########
from PIL import Image
from torchvision import transforms

# Load and normalize image
img = Image.open("/content/drive/MyDrive/20251006decoded_eval/20251006Region9RecoveredNogeneration.png").convert("RGB")
img_tensor = transforms.ToTensor()(img)  # shape [3, H, W], in [0,1]
ae = Autoencoder().to("cuda")
ae.load_state_dict(torch.load("/content/drive/MyDrive/newae2.pth", map_location="cuda"))
ae.eval()
# Predict cell composition distribution - Zach use to get data
cellmap = cell_map(img_tensor, ae)


Image RGB min/max:
  R: 0.0470588244497776 1.0
  G: 0.054901961237192154 1.0
  B: 0.05882352963089943 1.0
Valid pixels: 44021
Composition distribution: [0.02396583 0.06101633 0.00756457 0.02362509 0.00951818 0.07128416
 0.04363826 0.01749165 0.01830944 0.15513051 0.03393835 0.01367529
 0.06517344 0.04232071 0.05376979 0.01694646 0.02712342 0.01428863
 0.01494741 0.00870039 0.03811817 0.060562   0.06942141 0.07323777
 0.03623271]


In [ ]:
# === Bulk region processing via your AE decoder -> per-pixel cell types ===
import os
from typing import Iterable, List
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torchvision import transforms

# --------- Paths / config ---------
PNG_DIR = "/content/drive/MyDrive/vae-decoded"   # where region_*_recon.png live
SAVE_DIR = "/content/drive/MyDrive/vae_decoded_csv"             # where CSVs will go
os.makedirs(SAVE_DIR, exist_ok=True)

# If you want to limit images, adjust this range (inclusive of start, inclusive of end with +1)
REGION_RANGE = range(0, 66)  # 0..65

# Assume ae_model is already constructed + weights loaded
# ae_model = ... ; ae_model.eval()

# --------- Device setup ---------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --------- Optional: speed tweaks ---------
torch.set_grad_enabled(False)
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True

# --------- Your function (device-agnostic + minor return tweak) ---------
def cell_map(
    image_tensor: torch.Tensor,
    ae_model: torch.nn.Module,
    num_types: int = 25,
    device: str = DEVICE
) -> torch.Tensor:
    """
    Convert an RGB image tensor to a per-pixel cell-type map using the AE decoder.

    Args:
        image_tensor (torch.Tensor): [3, H, W] in [0,1]
        ae_model (torch.nn.Module): has .decoder(z) -> logits [N, num_types]
        num_types (int): number of cell types (default 25)
        device (str): 'cuda' or 'cpu'

    Returns:
        torch.LongTensor of shape [H, W], values:
            0 for white/ignored pixels,
            1..num_types for predicted cell types
    """
    assert image_tensor.ndim == 3 and image_tensor.shape[0] == 3, "Expecting [3, H, W]"
    image_tensor = image_tensor.to(device, non_blocking=True)

    # Debug (optional; comment out if noisy)
    # print("Image RGB min/max:",
    #       image_tensor[0].amin().item(), image_tensor[0].amax().item(),
    #       image_tensor[1].amin().item(), image_tensor[1].amax().item(),
    #       image_tensor[2].amin().item(), image_tensor[2].amax().item())

    # Constants to recover z from RGB (your values)
    min_vals = torch.tensor([-69.761505, -75.65188,  -77.16103], device=device)
    max_vals = torch.tensor([ 88.969406,  65.244896,  67.13518], device=device)
    range_vals = max_vals - min_vals  # elementwise

    # Reshape
    H, W = image_tensor.shape[1], image_tensor.shape[2]
    flat_img = image_tensor.permute(1, 2, 0).reshape(-1, 3)  # [H*W, 3]

    # White mask (ignore white pixels)
    white_mask = (flat_img > 0.95).all(dim=1)

    pred = torch.zeros(flat_img.shape[0], dtype=torch.long, device=device)

    num_valid = (~white_mask).sum()
    if num_valid > 0:
        infer_input_rgb = flat_img[~white_mask]  # [N, 3] in [0,1]
        z_recovered = infer_input_rgb * range_vals + min_vals  # [N, 3]

        # Decode → logits [N, num_types]
        logits = ae_model.decoder(z_recovered)

        # Map to 1..num_types
        pred[~white_mask] = torch.argmax(logits, dim=1) + 1

    # Back to [H, W] on CPU
    pred_map = pred.reshape(H, W).to("cpu")
    return pred_map

# --------- Image -> tensor helper ---------
to_tensor = transforms.ToTensor()  # returns [0,1] float tensor, shape [C,H,W]

def load_region_as_tensor(png_path: str) -> torch.Tensor:
    with Image.open(png_path) as im:
        im = im.convert("RGB")
        return to_tensor(im)  # [3,H,W], float32 in [0,1]

# --------- Main processing ---------
def process_regions(
    ae_model: torch.nn.Module,
    indices: Iterable[int] = REGION_RANGE,
    png_dir: str = PNG_DIR,
    save_dir: str = SAVE_DIR,
    num_types: int = 25,
) -> str:
    """
    Runs cell_map on region_{i}.png for i in indices.
    Saves per-region CSVs and a combined CSV. Returns combined path.
    """
    ae_model.eval()
    ae_model.to(DEVICE)

    combined_rows: List[pd.DataFrame] = []

    for i in indices:
        fname = f"region_{i}_recon.png"
        path = os.path.join(png_dir, fname)
        if not os.path.exists(path):
            print(f"[WARN] Missing: {path} — skipping.")
            continue

        # Load -> map
        img_t = load_region_as_tensor(path)  # [3,H,W]
        pred_map = cell_map(img_t, ae_model, num_types=num_types, device=DEVICE)  # [H,W] long

        # Keep only non-zero (non-white) pixels
        mask = pred_map.numpy() > 0
        if not mask.any():
            print(f"[INFO] {fname}: no non-white pixels found.")
            # Still write an empty CSV for consistency
            empty_df = pd.DataFrame(columns=["region", "x", "y", "cell_type"])
            per_csv = os.path.join(save_dir, f"region_{i}_cells.csv")
            empty_df.to_csv(per_csv, index=False)
            continue

        ys, xs = np.nonzero(mask)             # row (y), col (x)
        types = pred_map.numpy()[ys, xs]      # 1..num_types

        df = pd.DataFrame({
            "region": i,
            "x": xs.astype(np.int32),         # x = column index
            "y": ys.astype(np.int32),         # y = row index
            "cell_type": types.astype(np.int16)
        })

        # Per-region CSV
        per_csv = os.path.join(save_dir, f"region_{i}_cells.csv")
        df.to_csv(per_csv, index=False)
        print(f"[OK] Wrote {per_csv} ({len(df):,} rows)")

        combined_rows.append(df)

    # Combined CSV
    if combined_rows:
        combined = pd.concat(combined_rows, ignore_index=True)
    else:
        combined = pd.DataFrame(columns=["region", "x", "y", "cell_type"])

    combined_csv = os.path.join(save_dir, "all_regions_cells.csv")
    combined.to_csv(combined_csv, index=False)
    print(f"[DONE] Wrote combined CSV: {combined_csv} ({len(combined):,} total rows)")
    return combined_csv

# --------- Run it ---------
ae_model = Autoencoder().to("cuda")
ae_model.load_state_dict(torch.load("/content/drive/MyDrive/newae2.pth", map_location="cuda"))
ae_model.eval()
combined_csv_path = process_regions(ae_model)


[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_0_cells.csv (16,434 rows)
[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_1_cells.csv (25,432 rows)
[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_2_cells.csv (20,036 rows)
[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_3_cells.csv (34,137 rows)
[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_4_cells.csv (19,913 rows)
[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_5_cells.csv (27,583 rows)
[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_6_cells.csv (22,333 rows)
[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_7_cells.csv (26,803 rows)
[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_8_cells.csv (14,828 rows)
[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_9_cells.csv (48,322 rows)
[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_10_cells.csv (11,846 rows)
[OK] Wrote /content/drive/MyDrive/vae_decoded_csv/region_11_cells.csv (20,175 rows)
[O

In [ ]:
## TODO
##LEYLA: ADD PATH TO GENERATED LATENTS FROM GAP FILLER
##

##YUAN: CREATE PIPLELINE FOR INFERENCE ON GAP FILLER LATENTS

In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms
from diffusers import AutoencoderKL, DDPMScheduler
from accelerate import Accelerator

from google.colab import drive
drive.mount('/content/drive')


############## MAIN FUNCTION ENTRANCE(from latent to image) #############

latent_path = "5_latent.pt"
save_dir = "./"
vae_path = "runwayml/stable-diffusion-v1-5"
ckpt_path = "/content/drive/MyDrive/checkpoint_cascade512_best1/"
os.makedirs(save_dir, exist_ok=True)

# ==== Set up device and accelerator ====
device = torch.device("cuda")
accelerator = Accelerator(mixed_precision="fp16")

# ==== Load diffusion model components ====
adapter = LatentAdapter(cz=4, cond_ch=96)
unet = UNet512(base_ch=192, cond_ch=96, time_dim=256)
scheduler = DDPMScheduler.from_pretrained(vae_path, subfolder="scheduler")
scheduler.set_timesteps(150, device=device)
scheduler.config.prediction_type = "sample"

# ==== Prepare and load checkpoint ====
adapter, unet = accelerator.prepare(adapter, unet)
accelerator.load_state(ckpt_path)
adapter.eval(); unet.eval()

# ==== Load latent feature ====
z = torch.load(latent_path, map_location="cpu").to(device=device, dtype=torch.float16)
if z.ndim == 3:
    z = z.unsqueeze(0)  # ensure shape [1,4,64,64]

# ==== Decode with diffusion ====
with torch.no_grad():
    cond_feats = adapter(z)
    B, _, H, W = 1, 3, 512, 512
    x = torch.randn(B, 3, H, W, device=device) * scheduler.init_noise_sigma

    for t in scheduler.timesteps:
        t_batch = torch.full((B,), int(t), device=device, dtype=torch.long)
        x0_pred = unet(x, t_batch, cond_feats)
        x = scheduler.step(x0_pred, t, x).prev_sample

    out_img = (x.clamp(-1, 1) + 1) / 2  # to [0,1]

# ==== Save as PNG ====
to_pil = transforms.ToPILImage()
out_pil = to_pil(out_img[0].float().cpu())
out_pil.save(os.path.join(save_dir, "decoded_from_latent.png"))
print(f"Saved decoded PNG to: {save_dir}/decoded_from_latent.png")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

Saved decoded PNG to: .//decoded_from_latent.png


In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms
from diffusers import DDPMScheduler
from accelerate import Accelerator

############## MAIN FUNCTION ENTRANCE(from latent to image) #############

latent_root = "drive/MyDrive/outinfer/drive/MyDrive/outtest"   # 根目录，里面有多个子文件夹
save_root = "drive/MyDrive/outinfer/outdecoded"
vae_path = "runwayml/stable-diffusion-v1-5"
ckpt_path = "/content/drive/MyDrive/checkpoint_cascade512_best1"
os.makedirs(save_root, exist_ok=True)

# ==== Set up device and accelerator ====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
accelerator = Accelerator(mixed_precision="fp16")

# ==== Load diffusion model components ====
adapter = LatentAdapter(cz=4, cond_ch=96)
unet = UNet512(base_ch=192, cond_ch=96, time_dim=256)
scheduler = DDPMScheduler.from_pretrained(vae_path, subfolder="scheduler")
scheduler.set_timesteps(150, device=device)
scheduler.config.prediction_type = "sample"

# ==== Load checkpoint ====
adapter, unet = accelerator.prepare(adapter, unet)
accelerator.load_state(ckpt_path)
adapter.eval(); unet.eval()

to_pil = transforms.ToPILImage()

# ==== 遍历所有子文件夹 ====
target_name = "04_latent.pt"
latent_paths = []

for root, dirs, files in os.walk(latent_root):
    if target_name in files:
        latent_paths.append(os.path.join(root, target_name))

print(f"✅ Found {len(latent_paths)} latent files named '{target_name}' in subfolders.\n")

# ==== Decode each latent ====
for i, latent_path in enumerate(latent_paths):
    # 构造输出路径（与子文件夹结构对应）
    rel_dir = os.path.relpath(os.path.dirname(latent_path), latent_root)
    save_dir = os.path.join(save_root, rel_dir)
    os.makedirs(save_dir, exist_ok=True)

    save_path = os.path.join(save_dir, "04_image.png")
    print(f"[{i+1}/{len(latent_paths)}] Decoding {latent_path} -> {save_path}")

    # --- Load latent ---
    z = torch.load(latent_path, map_location="cpu").to(device=device, dtype=torch.float16)
    if z.ndim == 3:
        z = z.unsqueeze(0)  # ensure shape [1,4,H,W]

    # --- Decode using diffusion ---
    with torch.no_grad():
        cond_feats = adapter(z)
        B, _, H, W = 1, 3, 512, 512
        x = torch.randn(B, 3, H, W, device=device) * scheduler.init_noise_sigma

        for t in scheduler.timesteps:
            t_batch = torch.full((B,), int(t), device=device, dtype=torch.long)
            x0_pred = unet(x, t_batch, cond_feats)
            x = scheduler.step(x0_pred, t, x).prev_sample

        out_img = (x.clamp(-1, 1) + 1) / 2  # scale to [0,1]

    # --- Save as PNG ---
    out_pil = to_pil(out_img[0].float().cpu())
    out_pil.save(save_path)
    print(f"✅ Saved: {save_path}")

print(f"\n🎉 All '{target_name}' latents decoded and saved successfully under '{save_root}'.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

✅ Found 42 latent files named '04_latent.pt' in subfolders.

[1/42] Decoding drive/MyDrive/outinfer/drive/MyDrive/outtest/1.png1/04_latent.pt -> drive/MyDrive/outinfer/outdecoded/1.png1/04_image.png
✅ Saved: drive/MyDrive/outinfer/outdecoded/1.png1/04_image.png
[2/42] Decoding drive/MyDrive/outinfer/drive/MyDrive/outtest/12.png0/04_latent.pt -> drive/MyDrive/outinfer/outdecoded/12.png0/04_image.png
✅ Saved: drive/MyDrive/outinfer/outdecoded/12.png0/04_image.png
[3/42] Decoding drive/MyDrive/outinfer/drive/MyDrive/outtest/12.png2/04_latent.pt -> drive/MyDrive/outinfer/outdecoded/12.png2/04_image.png
✅ Saved: drive/MyDrive/outinfer/outdecoded/12.png2/04_image.png
[4/42] Decoding drive/MyDrive/outinfer/drive/MyDrive/outtest/10.png0/04_latent.pt -> drive/MyDrive/outinfer/outdecoded/10.png0/04_image.png
✅ Saved: drive/MyDrive/outinfer/outdecoded/10.png0/04_image.png
[5/42] Decoding drive/MyDrive/outinfer/drive/MyDrive/outtest/10.png1/04_latent.pt -> drive/MyDrive/outinfer/outdecoded/10.png1

In [6]:
import os
import torch
from PIL import Image
from torchvision import transforms
from diffusers import DDPMScheduler
from accelerate import Accelerator

############## MAIN FUNCTION (Decode all .pt latents to images) #############

latent_root = "./"
save_root = "drive/MyDrive/merfishinpaint"
vae_path = "runwayml/stable-diffusion-v1-5"
ckpt_path = "/content/drive/MyDrive/checkpoint_cascade512_best_m"
os.makedirs(save_root, exist_ok=True)

# ==== Setup device and accelerator ====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
accelerator = Accelerator(mixed_precision="fp16")

# ==== Load model components ====
adapter = LatentAdapter(cz=4, cond_ch=96)
unet = UNet512(base_ch=192, cond_ch=96, time_dim=256)
scheduler = DDPMScheduler.from_pretrained(vae_path, subfolder="scheduler")
scheduler.set_timesteps(150, device=device)
scheduler.config.prediction_type = "sample"

adapter, unet = accelerator.prepare(adapter, unet)
accelerator.load_state(ckpt_path)
adapter.eval(); unet.eval()

to_pil = transforms.ToPILImage()

# ==== Recursively collect all .pt files ====
latent_paths = []
for root, dirs, files in os.walk(latent_root):
    for f in files:
        if f.endswith(".pt"):
            latent_paths.append(os.path.join(root, f))

print(f"✅ Found {len(latent_paths)} .pt latent files in '{latent_root}'\n")

# ==== Decode each latent ====
for i, latent_path in enumerate(latent_paths):
    rel_dir = os.path.relpath(os.path.dirname(latent_path), latent_root)
    save_dir = os.path.join(save_root, rel_dir)
    os.makedirs(save_dir, exist_ok=True)

    latent_name = os.path.splitext(os.path.basename(latent_path))[0]
    save_path = os.path.join(save_dir, f"{latent_name}.png")
    print(f"[{i+1}/{len(latent_paths)}] Decoding {latent_path} -> {save_path}")

    # --- Load latent ---
    z_raw = torch.load(latent_path, map_location="cpu")
    if isinstance(z_raw, dict) and "blended_latent" in z_raw:
        z = z_raw["blended_latent"]
    else:
        z = z_raw  # assume it's directly a tensor

    z = z.to(device=device, dtype=torch.float16)
    if z.ndim == 3:
        z = z.unsqueeze(0)

    # --- Decode using diffusion ---
    with torch.no_grad():
        cond_feats = adapter(z)
        B, _, H, W = 1, 3, 512, 512
        x = torch.randn(B, 3, H, W, device=device) * scheduler.init_noise_sigma

        for t in scheduler.timesteps:
            t_batch = torch.full((B,), int(t), device=device, dtype=torch.long)
            x0_pred = unet(x, t_batch, cond_feats)
            x = scheduler.step(x0_pred, t, x).prev_sample

        out_img = (x.clamp(-1, 1) + 1) / 2

    # --- Save image ---
    out_pil = to_pil(out_img[0].float().cpu())
    out_pil.save(save_path)
    print(f"✅ Saved: {save_path}")

print(f"\n🎉 All latent .pt files decoded and saved under '{save_root}'")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

✅ Found 8574 .pt latent files in './'

[1/8574] Decoding ./6_size4_sample1_latent.pt -> drive/MyDrive/merfishinpaint/./6_size4_sample1_latent.png
✅ Saved: drive/MyDrive/merfishinpaint/./6_size4_sample1_latent.png
[2/8574] Decoding ./5_size4_sample1_latent.pt -> drive/MyDrive/merfishinpaint/./5_size4_sample1_latent.png
✅ Saved: drive/MyDrive/merfishinpaint/./5_size4_sample1_latent.png
[3/8574] Decoding ./7_size4_sample1_latent.pt -> drive/MyDrive/merfishinpaint/./7_size4_sample1_latent.png
✅ Saved: drive/MyDrive/merfishinpaint/./7_size4_sample1_latent.png
[4/8574] Decoding ./4_size3_sample1_latent.pt -> drive/MyDrive/merfishinpaint/./4_size3_sample1_latent.png
✅ Saved: drive/MyDrive/merfishinpaint/./4_size3_sample1_latent.png
[5/8574] Decoding ./3_size4_sample1_latent.pt -> drive/MyDrive/merfishinpaint/./3_size4_sample1_latent.png
✅ Saved: drive/MyDrive/merfishinpaint/./3_size4_sample1_latent.png
[6/8574] Decoding ./6_size3_sample1_latent.pt -> drive/MyDrive/merfishinpaint/./6_size3_sam

AttributeError: 'dict' object has no attribute 'to'

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from diffusers import AutoencoderKL, DDPMScheduler
from accelerate import Accelerator
from torchvision import transforms
from scipy.spatial.distance import jensenshannon

# ---------- Helper Functions ----------

def compute_jsd(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p /= p.sum(); q /= q.sum()
    return jensenshannon(p, q, base=2) ** 2

def compute_type_distribution(type_map_np, num_types=25):
    counts = np.bincount(type_map_np.ravel(), minlength=num_types + 1)[1:num_types + 1].astype(float)
    total = counts.sum()
    if total <= 0:
        return np.zeros(num_types, dtype=float)
    return counts / total

def infer_cell_map(latent_image, model):
    latent_image = latent_image.half().to("cuda")
    min_vals = torch.tensor([-69.761505, -75.65188,  -77.16103], device=latent_image.device)
    range_vals = torch.tensor([88.969406, 65.244896, 67.13518], device=latent_image.device) - min_vals
    H, W = latent_image.shape[1], latent_image.shape[2]
    flat_img = latent_image.permute(1, 2, 0).reshape(-1, 3)
    white_mask = (flat_img > 0.93).all(dim=1)
    infer_input_rgb = flat_img[~white_mask]
    pred = torch.zeros(flat_img.shape[0], dtype=torch.long, device="cuda")
    if infer_input_rgb.shape[0] > 0:
        z_recovered = infer_input_rgb * range_vals + min_vals
        logits = model.decoder(z_recovered)
        pred[~white_mask] = torch.argmax(logits, dim=1) + 1
    return pred.reshape(1, H, W)

# ---------- Main Function ----------

@torch.no_grad()
def decode_and_eval_patch_distributions(
    root_dir, save_dir,
    vae_path="runwayml/stable-diffusion-v1-5",
    ckpt_path="/content/drive/MyDrive/checkpoint_cascade512_best1",
    steps=100
):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda")
    accelerator = Accelerator(mixed_precision="fp16")

    # Load model
    adapter = LatentAdapter(cz=4, cond_ch=96)
    unet = UNet512(base_ch=192, cond_ch=96, time_dim=256)
    scheduler = DDPMScheduler.from_pretrained(vae_path, subfolder="scheduler")
    scheduler.set_timesteps(steps, device=device)
    scheduler.config.prediction_type = "sample"

    adapter, unet = accelerator.prepare(adapter, unet)
    accelerator.load_state(ckpt_path)
    adapter.eval(); unet.eval()

    ae_model = Autoencoder().to(device)
    ae_model.load_state_dict(torch.load("drive/MyDrive/newae2.pth", map_location=device))
    ae_model.eval()

    # Start processing regions
    region_dirs = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    top_dists, bot_dists, region_names = [], [], []

    for region in tqdm(region_dirs, desc="Processing regions"):
        latent_path = os.path.join(root_dir, region, "4_latent.pt")
        if not os.path.exists(latent_path):
            continue

        z = torch.load(latent_path, map_location="cpu").to(device, dtype=torch.float16)
        if z.ndim == 3:
            z = z.unsqueeze(0)

        # Decode with diffusion
        cond_feats = adapter(z)
        B, _, H, W = 1, 3, 512, 512
        x = torch.randn(B, 3, H, W, device=device) * scheduler.init_noise_sigma
        for t in scheduler.timesteps:
            t_batch = torch.full((B,), int(t), device=device, dtype=torch.long)
            x0_pred = unet(x, t_batch, cond_feats)
            x = scheduler.step(x0_pred, t, x).prev_sample
        img = (x.clamp(-1, 1) + 1) / 2  # [1,3,H,W]

        img_top = img[0, :, :int(H * 0.3), :]
        img_bot = img[0, :, int(H * 0.3):, :]

        # --- Filter if top is too white
        flat_top = img_top.permute(1, 2, 0).reshape(-1, 3)
        white_mask = (flat_top > 0.93).all(dim=1)
        white_ratio = white_mask.float().mean().item()
        if white_ratio > 0.9:
            print(f"⚠️  Skipping {region} due to >90% white in top patch ({white_ratio:.2%})")
            continue

        # Inference
        type_top = infer_cell_map(img_top, model=ae_model)
        type_bot = infer_cell_map(img_bot, model=ae_model)

        dist_top = compute_type_distribution(type_top.squeeze(0).cpu().numpy(), num_types=25)
        dist_bot = compute_type_distribution(type_bot.squeeze(0).cpu().numpy(), num_types=25)

        top_dists.append(dist_top)
        bot_dists.append(dist_bot)
        region_names.append(region)

    # Build JSD matrices
    top_dists = np.stack(top_dists, axis=0)
    bot_dists = np.stack(bot_dists, axis=0)
    N = len(top_dists)

    top_jsd = np.zeros((N, N))
    bot_jsd = np.zeros((N, N))
    cross_jsd = np.zeros((N, N))

    for i in range(N):
        for j in range(N):
            top_jsd[i, j] = compute_jsd(top_dists[i], top_dists[j])
            bot_jsd[i, j] = compute_jsd(bot_dists[i], bot_dists[j])
            cross_jsd[i, j] = compute_jsd(top_dists[i], bot_dists[j])

    # Save heatmaps
    def plot_heatmap(mat, title, fname):
        plt.figure(figsize=(0.4 * N + 4, 0.4 * N + 4))
        sns.heatmap(mat, xticklabels=region_names, yticklabels=region_names,
                    cmap="plasma", annot=False, vmin=0, vmax=1)
        plt.title(title)
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, fname), dpi=200)
        plt.close()

    plot_heatmap(top_jsd, "Top Patch JSD", "jsd_top.png")
    plot_heatmap(bot_jsd, "Bottom Patch JSD", "jsd_bottom.png")
    plot_heatmap(cross_jsd, "Cross Patch JSD", "jsd_cross.png")

    print(f"[Done] Saved all heatmaps to {save_dir}")


In [ ]:
import os
import re
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from diffusers import DDPMScheduler
from accelerate import Accelerator
from scipy.spatial.distance import jensenshannon

# ========= 工具函数 =========

def compute_jsd(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p /= p.sum(); q /= q.sum()
    return jensenshannon(p, q, base=2) ** 2

def compute_type_distribution(type_map_np, num_types=25):
    counts = np.bincount(type_map_np.ravel(), minlength=num_types + 1)[1:num_types + 1].astype(float)
    total = counts.sum()
    if total <= 0:
        return np.zeros(num_types, dtype=float)
    return counts / total

def infer_cell_map(latent_image, model):
    latent_image = latent_image.half().to("cuda")
    min_vals = torch.tensor([-69.761505, -75.65188,  -77.16103], device=latent_image.device)
    range_vals = torch.tensor([88.969406, 65.244896, 67.13518], device=latent_image.device) - min_vals
    H, W = latent_image.shape[1], latent_image.shape[2]
    flat_img = latent_image.permute(1, 2, 0).reshape(-1, 3)
    white_mask = (flat_img > 0.95).all(dim=1)
    infer_input_rgb = flat_img[~white_mask]
    pred = torch.zeros(flat_img.shape[0], dtype=torch.long, device=latent_image.device)
    if infer_input_rgb.shape[0] > 0:
        z_recovered = infer_input_rgb * range_vals + min_vals
        logits = model.decoder(z_recovered)
        pred[~white_mask] = torch.argmax(logits, dim=1) + 1
    return pred.reshape(1, H, W)

# ========= 主函数 =========

@torch.no_grad()
def decode_and_eval_patch_distributions(
    root_dir, save_dir, steps=100
):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda")
    # Load model
    accelerator = Accelerator(mixed_precision="fp16")
    adapter = LatentAdapter(cz=4, cond_ch=96)
    unet = UNet512(base_ch=192, cond_ch=96, time_dim=256)
    scheduler = DDPMScheduler.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="scheduler")
    scheduler.set_timesteps(steps, device=device)
    scheduler.config.prediction_type = "sample"

    adapter, unet = accelerator.prepare(adapter, unet)
    accelerator.load_state("/content/drive/MyDrive/checkpoint_cascade512_best1")
    adapter.eval(); unet.eval()

    ae_model = Autoencoder().to(device)
    ae_model.load_state_dict(torch.load("drive/MyDrive/newae2.pth", map_location=device))
    ae_model.eval()

    # Step 1. 读取 region 文件夹并按 prefix 分组
    region_dirs = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    pattern = re.compile(r"^(\d+)\+(\d+)$")

    grouped = {}
    for d in region_dirs:
        match = pattern.match(d)
        if match:
            prefix = match.group(1)
            grouped.setdefault(prefix, []).append(d)

    print(f"Found {len(grouped)} prefix groups.")

    selected_regions = []  # 保存选中的 region
    top_dists, bot_dists, region_names = [], [], []

    for prefix, subdirs in tqdm(grouped.items(), desc="Selecting 1 sample per prefix"):
        subdirs = sorted(subdirs)
        found_valid = False

        for region in subdirs:
            latent_path = os.path.join(root_dir, region, "4_latent.pt")
            if not os.path.exists(latent_path):
                continue

            z = torch.load(latent_path, map_location="cpu").to(device, dtype=torch.float16)
            if z.ndim == 3:
                z = z.unsqueeze(0)

            # --- Diffusion decode ---
            cond_feats = adapter(z)
            B, _, H, W = 1, 3, 512, 512
            x = torch.randn(B, 3, H, W, device=device) * scheduler.init_noise_sigma
            for t in scheduler.timesteps:
                t_batch = torch.full((B,), int(t), device=device, dtype=torch.long)
                x0_pred = unet(x, t_batch, cond_feats)
                x = scheduler.step(x0_pred, t, x).prev_sample
            img = (x.clamp(-1, 1) + 1) / 2  # [1,3,H,W]

            # --- top/bottom split ---
            img_top = img[0, :, :int(H * 0.3), :]
            img_bot = img[0, :, int(H * 0.3):, :]

            # --- 白像素过滤 ---
            flat_top = img_top.permute(1, 2, 0).reshape(-1, 3)
            white_mask = (flat_top > 0.93).all(dim=1)
            white_ratio = white_mask.float().mean().item()
            if white_ratio > 0.9:
                print(f"⚠️  Skipping {region} due to >90% white top patch ({white_ratio:.2%})")
                continue

            # --- composition inference ---
            type_top = infer_cell_map(img_top, model=ae_model)
            type_bot = infer_cell_map(img_bot, model=ae_model)

            dist_top = compute_type_distribution(type_top.squeeze(0).cpu().numpy(), num_types=25)
            dist_bot = compute_type_distribution(type_bot.squeeze(0).cpu().numpy(), num_types=25)

            top_dists.append(dist_top)
            bot_dists.append(dist_bot)
            region_names.append(region)
            selected_regions.append(region)
            found_valid = True
            break  # ✅ 每个 prefix 只取一个样本

        if not found_valid:
            print(f"❌ No valid (non-white) sample found for prefix {prefix}")

    print(f"✅ Selected {len(selected_regions)} regions: {selected_regions}")

    # Step 2. 计算 Heatmap
    N = len(top_dists)
    top_dists = np.stack(top_dists, axis=0)
    bot_dists = np.stack(bot_dists, axis=0)

    top_jsd = np.zeros((N, N))
    bot_jsd = np.zeros((N, N))
    cross_jsd = np.zeros((N, N))

    for i in range(N):
        for j in range(N):
            top_jsd[i, j] = compute_jsd(top_dists[i], top_dists[j])
            bot_jsd[i, j] = compute_jsd(bot_dists[i], bot_dists[j])
            cross_jsd[i, j] = compute_jsd(top_dists[i], bot_dists[j])

    def plot_heatmap(mat, title, fname):
        plt.figure(figsize=(0.4 * N + 4, 0.4 * N + 4))
        sns.heatmap(mat, xticklabels=region_names, yticklabels=region_names,
                    cmap="plasma", annot=False, vmin=0, vmax=1)
        plt.title(title)
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, fname), dpi=200)
        plt.close()

    plot_heatmap(top_jsd, "Top Patch JSD", "jsd_top.png")
    plot_heatmap(bot_jsd, "Bottom Patch JSD", "jsd_bottom.png")
    plot_heatmap(cross_jsd, "Cross Patch JSD", "jsd_cross.png")

    print(f"🔥 Done. Saved all heatmaps to {save_dir}")


In [ ]:
decode_and_eval_patch_distributions(
    root_dir="drive/MyDrive/longinfer",
    save_dir="drive/MyDrive/longinfer/eval_patch"
)

Found 39 prefix groups.


Selecting 1 sample per prefix:   3%|▎         | 1/39 [00:33<21:08, 33.37s/it]

⚠️  Skipping 0+9 due to >90% white top patch (90.59%)
❌ No valid (non-white) sample found for prefix 0


Selecting 1 sample per prefix:  13%|█▎        | 5/39 [02:38<17:50, 31.49s/it]

⚠️  Skipping 5+0 due to >90% white top patch (95.66%)
⚠️  Skipping 5+1 due to >90% white top patch (95.81%)
⚠️  Skipping 5+2 due to >90% white top patch (95.92%)
⚠️  Skipping 5+3 due to >90% white top patch (94.90%)
⚠️  Skipping 5+4 due to >90% white top patch (96.17%)
⚠️  Skipping 5+5 due to >90% white top patch (96.06%)
⚠️  Skipping 5+6 due to >90% white top patch (95.21%)
⚠️  Skipping 5+7 due to >90% white top patch (96.23%)
⚠️  Skipping 5+8 due to >90% white top patch (95.97%)


...

⚠️  Skipping 37+0 due to >90% white top patch (93.14%)
⚠️  Skipping 37+1 due to >90% white top patch (90.38%)
⚠️  Skipping 37+2 due to >90% white top patch (92.02%)
⚠️  Skipping 37+3 due to >90% white top patch (90.61%)
⚠️  Skipping 37+4 due to >90% white top patch (96.56%)
⚠️  Skipping 37+5 due to >90% white top patch (93.23%)
⚠️  Skipping 37+6 due to >90% white top patch (93.86%)
⚠️  Skipping 37+7 due to >90% white top patch (92.40%)
⚠️  Skipping 37+8 due to >90% white top patch (93.68%)


Selecting 1 sample per prefix:  97%|█████████▋| 38/39 [1:16:59<02:46, 166.35s/it]

⚠️  Skipping 37+9 due to >90% white top patch (93.07%)
❌ No valid (non-white) sample found for prefix 37


Selecting 1 sample per prefix: 100%|██████████| 39/39 [1:17:30<00:00, 119.25s/it]


✅ Selected 28 regions: ['1+5', '2+0', '3+0', '4+0', '6+1', '7+0', '9+0', '10+0', '11+1', '13+0', '14+5', '15+3', '16+1', '17+0', '19+0', '20+0', '22+0', '23+0', '25+3', '27+4', '29+0', '30+0', '31+1', '32+0', '33+0', '34+0', '36+0', '38+0']
🔥 Done. Saved all heatmaps to drive/MyDrive/longinfer/eval_patch


In [ ]:
# save_as: decode_gap_infer_and_heatmaps.py
import os
import re
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from diffusers import DDPMScheduler
from accelerate import Accelerator
from scipy.spatial.distance import jensenshannon
from torchvision import transforms
from PIL import Image

# -------------------------
# User must provide/define/ import these:
#   LatentAdapter, UNet512, Autoencoder
#   - LatentAdapter: maps latent z_cond -> cond_feats for UNet512
#   - UNet512: your cascade unet (predicts x0)
#   - Autoencoder: your classifier/decoder with .decoder() used to infer cell types
# -------------------------
# from mymodels import LatentAdapter, UNet512, Autoencoder   # <-- replace with your import

# -------------------------
# Utility functions
# -------------------------
def compute_jsd(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    # avoid zero-sum
    if p.sum() == 0:
        p = np.ones_like(p) / len(p)
    if q.sum() == 0:
        q = np.ones_like(q) / len(q)
    p /= p.sum(); q /= q.sum()
    return jensenshannon(p, q, base=2) ** 2

def compute_type_distribution(type_map_np, num_types=25):
    counts = np.bincount(type_map_np.ravel(), minlength=num_types + 1)[1:num_types + 1].astype(float)
    total = counts.sum()
    if total <= 0:
        return np.zeros(num_types, dtype=float)
    return counts / total

def infer_cell_map_local(img_rgb_01, classifier_model, device):
    """
    Infer cell map for a single patch.
    Args:
        img_rgb_01: torch.Tensor [3,H,W], values in [0,1], dtype can be float16/float32
        classifier_model: Autoencoder-like model with .decoder() that takes Nx3 z and outputs logits [N,25]
        device: device for classifier
    Returns:
        pred_map: torch.LongTensor shape [1,H,W] with values in {0..25} where 0 means background/white
    """
    # move & dtype safe: convert RGB in [0,1] -> float32 on classifier device
    img = img_rgb_01.to(device).float()  # [3,H,W]
    H, W = img.shape[1], img.shape[2]
    flat = img.permute(1,2,0).reshape(-1,3)  # [HW,3]
    white_mask = (flat > 0.93).all(dim=1)    # boolean mask
    valid = flat[~white_mask]                # [N_valid,3]
    pred = torch.zeros(flat.shape[0], dtype=torch.long, device=device)  # default 0 (background)
    if valid.shape[0] > 0:
        # convert from [0,1] RGB back to original z space expected by your classifier:
        # NOTE: this needs to match your earlier scaling: z = rgb * range + min
        min_vals = torch.tensor([-69.761505, -75.65188,  -77.16103], device=device, dtype=torch.float32)
        range_vals = (torch.tensor([88.969406, 65.244896, 67.13518], device=device, dtype=torch.float32)
                      - min_vals)
        z_recovered = valid * range_vals + min_vals  # [N_valid,3]
        logits = classifier_model.decoder(z_recovered)  # [N_valid, num_types]
        labels = torch.argmax(logits, dim=1).to(device) + 1  # 1..25
        pred[~white_mask] = labels
    pred_map = pred.reshape(1, H, W).cpu()
    return pred_map  # cpu LongTensor

# -------------------------
# Main processing function
# -------------------------
@torch.no_grad()
def decode_and_make_heatmaps(
    root_dir,
    save_dir,
    ckpt_path,
    steps=50,
    latent_name="5_latent.pt",
    device=torch.device("cuda"),
    scheduler_model_id="runwayml/stable-diffusion-v1-5"
):
    os.makedirs(save_dir, exist_ok=True)

    # Accelerator & models
    accelerator = Accelerator(mixed_precision="fp16")
    adapter = LatentAdapter(cz=4, cond_ch=96)
    unet = UNet512(base_ch=192, cond_ch=96, time_dim=256)

    scheduler = DDPMScheduler.from_pretrained(scheduler_model_id, subfolder="scheduler")
    scheduler.set_timesteps(steps, device=device)
    scheduler.config.prediction_type = "sample"

    # Prepare + load checkpoint
    adapter, unet = accelerator.prepare(adapter, unet)
    accelerator.load_state(ckpt_path)  # assumes you saved with accelerate saving
    adapter.eval(); unet.eval()

    # load classifier autoencoder (decoder-based)
    ae_model = Autoencoder().to(device)
    ae_model.load_state_dict(torch.load("drive/MyDrive/newae2.pth", map_location=device))
    ae_model.eval()

    # locate subdirectories under root_dir (each subdir corresponds to an XXX)
    subdirs = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    if len(subdirs) == 0:
        print("[WARN] No subdirectories found under", root_dir)
        return

    # We'll accumulate distributions
    left_dists = []
    mid_dists = []
    right_dists = []
    region_names = []

    # Process each region subdir
    for name in tqdm(subdirs, desc="Decoding latents"):
        latent_path = os.path.join(root_dir, name, latent_name)
        if not os.path.exists(latent_path):
            print(f"[WARN] missing latent {latent_path}, skip.")
            continue

        # load latent (CPU) then move to device/dtype expected by adapter/unet
        z = torch.load(latent_path, map_location="cpu")
        if z.ndim == 3:
            z = z.unsqueeze(0)
        z = z.to(accelerator.device, dtype=torch.float16)

        # diffusion decode -> image in [-1,1]
        cond_feats = adapter(z)
        B, _, H, W = 1, 3, 512, 512
        x = torch.randn(B, 3, H, W, device=accelerator.device) * scheduler.init_noise_sigma
        for t in scheduler.timesteps:
            t_batch = torch.full((B,), int(t), device=accelerator.device, dtype=torch.long)
            x0_pred = unet(x, t_batch, cond_feats)
            x = scheduler.step(x0_pred, t, x).prev_sample
        img01 = (x.clamp(-1, 1) + 1) / 2  # [1,3,H,W], in [0,1], dtype fp16

        # split left/mid/right
        left_w = int(W * 0.125)
        right_w = left_w
        mid_w = W - left_w - right_w
        # slices: left: [:left_w], mid: [left_w:left_w+mid_w], right: [-right_w:]
        img = img01[0]  # [3,H,W]
        img_left = img[:, :, :left_w]
        img_mid  = img[:, :, left_w:left_w+mid_w]
        img_right= img[:, :, -right_w:]

        # compute white ratio on left; if >90% white, skip this sample (as requested earlier)
        flat_left = img_left.permute(1,2,0).reshape(-1,3)

        # infer cell maps (convert patches to classifier device inside)
        type_left = infer_cell_map_local(img_left, classifier_model=ae_model, device=device)   # [1,H_left,W_left] cpu
        type_mid  = infer_cell_map_local(img_mid, classifier_model=ae_model, device=device)
        type_right= infer_cell_map_local(img_right, classifier_model=ae_model, device=device)

        # compute distributions
        d_left  = compute_type_distribution(type_left.squeeze(0).numpy(), num_types=25)
        d_mid   = compute_type_distribution(type_mid.squeeze(0).numpy(), num_types=25)
        d_right = compute_type_distribution(type_right.squeeze(0).numpy(), num_types=25)

        left_dists.append(d_left)
        mid_dists.append(d_mid)
        right_dists.append(d_right)
        region_names.append(name)

        # Also save decoded full image for inspection
        out_img = (img01[0].detach().cpu().float().numpy().transpose(1,2,0) * 255).astype(np.uint8)
        Image.fromarray(out_img).save(os.path.join(save_dir, f"{name}_decoded.png"))

    # Convert lists -> arrays
    if len(region_names) == 0:
        print("[ERROR] No valid samples processed (maybe all skipped).")
        return

    left_arr  = np.stack(left_dists, axis=0)   # [N,25]
    mid_arr   = np.stack(mid_dists, axis=0)
    right_arr = np.stack(right_dists, axis=0)
    N = left_arr.shape[0]

    # Build heatmaps
    left_jsd = np.zeros((N,N))
    right_jsd= np.zeros((N,N))
    mid_vs_left = np.zeros((N,N))
    mid_vs_right = np.zeros((N,N))

    for i in range(N):
        for j in range(N):
            left_jsd[i,j] = compute_jsd(left_arr[i], left_arr[j])
            right_jsd[i,j]= compute_jsd(right_arr[i], right_arr[j])
            mid_vs_left[i,j] = compute_jsd(mid_arr[i], left_arr[j])
            mid_vs_right[i,j]= compute_jsd(mid_arr[i], right_arr[j])

    def plot_and_save_heatmap(mat, title, fname):
        plt.figure(figsize=(0.35 * max(10,N) + 4, 0.35 * max(10,N) + 4))
        sns.heatmap(mat, xticklabels=region_names, yticklabels=region_names,
                    cmap="plasma", annot=False, vmin=0, vmax=1)
        plt.title(title)
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        plt.tight_layout()
        outp = os.path.join(save_dir, fname)
        plt.savefig(outp, dpi=200)
        plt.close()
        print("[Saved]", outp)

    plot_and_save_heatmap(left_jsd, "Left 12.5% JSD (left vs left)", "heatmap_left_vs_left.png")
    plot_and_save_heatmap(right_jsd,"Right 12.5% JSD (right vs right)","heatmap_right_vs_right.png")
    plot_and_save_heatmap(mid_vs_left,"Mid vs Left JSD","heatmap_mid_vs_left.png")
    plot_and_save_heatmap(mid_vs_right,"Mid vs Right JSD","heatmap_mid_vs_right.png")

    # ---------------------------
    # Stacked bar: split each full decoded image into 8 vertical stripes and compute composition
    # ---------------------------
    type_colors = np.array([np.array(rgb) / 255.0 for rgb in type_color_dict.values()])

    stripes_save_dir = os.path.join(save_dir, "stripes_barcharts")
    os.makedirs(stripes_save_dir, exist_ok=True)

    for idx, name in enumerate(region_names):
        decoded_path = os.path.join(save_dir, f"{name}_decoded.png")
        if not os.path.exists(decoded_path):
            continue
        pil = Image.open(decoded_path).convert("RGB")
        W = pil.width; H = pil.height
        stripe_w = W // 8
        stripe_fracs = []
        for s in range(8):
            x1 = s * stripe_w
            x2 = W if s == 7 else (s+1) * stripe_w
            crop = pil.crop((x1, 0, x2, H))
            t = transforms.ToTensor()(crop).to(device).unsqueeze(0)  # [1,3,H_s,W_s] in [0,1]
            patch = t[0]
            type_map = infer_cell_map_local(patch, classifier_model=ae_model, device=device)
            frac = compute_type_distribution(type_map.squeeze(0).cpu().numpy(), num_types=25)
            stripe_fracs.append(frac)

        stripe_fracs = np.stack(stripe_fracs, axis=0)  # [8,25]

        fig, ax = plt.subplots(figsize=(12, 6))
        x = np.arange(8)
        bottom = np.zeros(8)
        for k in range(25):
            vals = stripe_fracs[:, k]
            color = type_colors[k]
            ax.bar(
                x,
                vals,
                bottom=bottom,
                color=color,
                label=cell_type_names[k],
                edgecolor="black",
                linewidth=0.2
            )
            bottom += vals

        ax.set_xlabel("Stripe (Left → Right, total 8)")
        ax.set_ylabel("Fraction")
        ax.set_title(f"{name} - Cell Composition (8 vertical stripes)")
        ax.set_xticks(x)
        ax.set_xticklabels([f"S{i+1}" for i in range(8)], fontsize=9)
        ax.set_ylim(0, 1)
        ax.legend(
            loc="upper left",
            bbox_to_anchor=(1.02, 1),
            ncol=1,
            fontsize=7,
            frameon=False
        )
        plt.tight_layout()
        out_path = os.path.join(stripes_save_dir, f"{name}_stripe_composition.png")
        plt.savefig(out_path, dpi=200)
        plt.close()
        print(f"[Saved stripe chart] {out_path}")

    print("All done. Heatmaps + stripe charts saved to:", save_dir)
    return {
        "region_names": region_names,
        "left_jsd": left_jsd,
        "right_jsd": right_jsd,
        "mid_vs_left": mid_vs_left,
        "mid_vs_right": mid_vs_right
    }


In [ ]:
result = decode_and_make_heatmaps(
     root_dir="drive/MyDrive/outinfer/drive/MyDrive/outtest",
     save_dir="drive/MyDrive/out/eval_out",
     ckpt_path="/content/drive/MyDrive/checkpoint_cascade512_best1",
     steps=150,
     latent_name="04_latent.pt",
     device=torch.device("cuda")
 )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

Decoding latents: 100%|██████████| 183/183 [2:20:46<00:00, 46.15s/it]


[WARN] missing latent drive/MyDrive/gapinfer/eval_out/5_latent.pt, skip.
[Saved] drive/MyDrive/gapinfer/eval_out/heatmap_left_vs_left.png
[Saved] drive/MyDrive/gapinfer/eval_out/heatmap_right_vs_right.png
[Saved] drive/MyDrive/gapinfer/eval_out/heatmap_mid_vs_left.png
[Saved] drive/MyDrive/gapinfer/eval_out/heatmap_mid_vs_right.png
[Saved stripe chart] drive/MyDrive/gapinfer/eval_out/stripes_barcharts/4_proximal_jejunum_right_10_duodenum_left.png_stripe_composition.png
[Saved stripe chart] drive/MyDrive/gapinfer/eval_out/stripes_barcharts/4_proximal_jejunum_right_10_mid_jejunum_left.png_stripe_composition.png
[Saved stripe chart] drive/MyDrive/gapinfer/eval_out/stripes_barcharts/4_proximal_jejunum_right_11_proximal_jejunum_left.png_stripe_composition.png
[Saved stripe chart] drive/MyDrive/gapinfer/eval_out/stripes_barcharts/4_proximal_jejunum_right_11_trans_left.png_stripe_composition.png
[Saved stripe chart] drive/MyDrive/gapinfer/eval_out/stripes_barcharts/4_proximal_jejunum_right_12

## To get CSV files for x,y,cell_type for each region after having been passed through the encoder and decoder.

In [ ]:
# === PNG → (VAE encode → diffusion decode) → AE pixelwise classifier → CSV ===
import os, glob
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from accelerate import Accelerator
from diffusers import AutoencoderKL, DDPMScheduler

# ---------------------------------------------------------------------------
# User-provided/assumed symbols from your environment:
#   - Autoencoder() class + weights file (AE_WTS)
#   - infer_cell_map(img_tensor, ae)  # returns [1,H,W] with labels {0..25}
#   - save_inferred_map_to_df(type_map, region_id)  # returns DataFrame with x,y,"Cell Type"
#   - cell_type_names: list of length 25
# ---------------------------------------------------------------------------

# ---- Paths (edit as needed) ----
PNG_DIR   = "/content/drive/MyDrive/202509CURRENT_Diff_TRAIN_set"
SAVE_DIR  = "/content/decoded_regions_csv"
AE_WTS    = "/content/drive/MyDrive/newae2.pth"
VAE_PATH  = "runwayml/stable-diffusion-v1-5"
CKPT_PATH = "/content/drive/MyDrive/checkpoint_cascade512_best1"
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- Sanity: class names exist ----
assert 'cell_type_names' in globals() and len(cell_type_names) == 25, \
    "Define `cell_type_names` (length 25) before running this cell."

# ---- Devices / precision ----
device = torch.device("cuda")
accelerator = Accelerator(mixed_precision='fp16')

# ---- Load your AE (classifier head through decoder) ----
ae = Autoencoder().to(device).eval()
ae.load_state_dict(torch.load(AE_WTS, map_location=device))

# ---- Load VAE (encoder) ----
vae = AutoencoderKL.from_pretrained(VAE_PATH, subfolder="vae", torch_dtype=torch.float16).to(device)
vae.eval()

# ---- Load diffusion adapter + UNet + scheduler ----
adapter = LatentAdapter(cz=4, cond_ch=96)
unet    = UNet512(base_ch=192, cond_ch=96, time_dim=256)
scheduler = DDPMScheduler.from_pretrained(VAE_PATH, subfolder="scheduler")
scheduler.set_timesteps(50, device=device)
scheduler.config.prediction_type = "sample"

# Prepare & load weights for adapter/unet
adapter, unet = accelerator.prepare(adapter, unet)
accelerator.load_state(CKPT_PATH)
adapter.eval(); unet.eval()

# ---- Transforms ----
to_tensor01 = transforms.ToTensor()                           # PIL → [0,1] float32
to_diff_in  = transforms.Compose([                            # PIL → [-1,1] float16 @ 512
    transforms.Resize((512, 512), interpolation=Image.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ---- Diffusion reconstruction helper ----
@torch.inference_mode()
def reconstruct_via_diffusion(pil_img: Image.Image,
                              steps: int = 50,
                              seed: int | None = None) -> torch.Tensor:
    """
    Args:
        pil_img: RGB PIL image at arbitrary H×W
        steps:   number of DDPM steps (scheduler already set to 50 by default)
        seed:    optional seed for reproducibility
    Returns:
        x_rec_orig: torch.FloatTensor, shape [3,H_orig,W_orig], range [0,1], on CUDA
    """
    H0, W0 = pil_img.height, pil_img.width

    # 1) Encode with VAE at 512×512 in [-1,1], fp16
    img_512 = to_diff_in(pil_img).unsqueeze(0).to(device, dtype=torch.float16)  # [1,3,512,512]

    # Stochasticity controls
    if seed is not None:
        g = torch.Generator(device=device).manual_seed(seed)
    else:
        g = None

    # VAE encoder: latent z (N(μ,σ)) — using sample like your reference
    z = vae.encode(img_512).latent_dist.sample(generator=g)                     # [1,4,64,64]

    # 2) Condition features via adapter
    cond_feats = adapter(z)                                                     # e.g., [1,96,64,64] (per your model)

    # 3) DDPM sampling in pixel space (UNet predicts x0 directly; you set prediction_type="sample")
    B, C, H, W = 1, 3, 512, 512
    # init noise (same distribution as scheduler.init_noise_sigma)
    x = torch.randn((B, C, H, W), device=device, dtype=torch.float16, generator=g) * scheduler.init_noise_sigma

    # Optionally override number of steps per call
    if steps != len(scheduler.timesteps):
        scheduler.set_timesteps(steps, device=device)

    for t in scheduler.timesteps:
        t_batch = torch.full((B,), int(t), device=device, dtype=torch.long)
        x0_pred = unet(x, t_batch, cond_feats)                                  # model-specific forward
        x = scheduler.step(x0_pred, t, x).prev_sample

    # 4) Back to [0,1] and original resolution
    x_img = (x.clamp(-1, 1) + 1) / 2                                            # [0,1], fp16
    x_img = x_img.to(dtype=torch.float32)                                       # AE expects fp32
    if (H, W) != (H0, W0):
        x_img = torch.nn.functional.interpolate(x_img, size=(H0, W0), mode="bilinear", align_corners=False)
    return x_img[0]                                                             # [3,H0,W0], cuda, float32

# ---- Iterate all regions ----
png_paths = sorted(glob.glob(os.path.join(PNG_DIR, "region_*.png")))
written, skipped = [], []

@torch.inference_mode()
def process_one_image(png_path: str):
    base = os.path.basename(png_path)
    rid_str = os.path.splitext(base)[0].split('_')[-1]
    region_id = int(rid_str)  # raises if malformed

    # Load original RGB (keep original size for downstream)
    pil = Image.open(png_path).convert("RGB")

    # A) Reconstruct via VAE→diffusion
    #    To reduce randomness across runs, set e.g. seed=123; or leave None for full stochastic decoding.
    rec_tensor = reconstruct_via_diffusion(pil, steps=50, seed=None)  # [3,H,W], [0,1], float32, cuda

    # B) Classify per-pixel cell type with your AE pathway
    #    infer_cell_map previously consumed ToTensor(pil)→[0,1] float32. rec_tensor matches that contract.
    type_map = infer_cell_map(rec_tensor, ae)  # [1,H,W], labels with 0=background, 1..25 = classes

    # C) Convert to your dataframe schema and save
    df_region = save_inferred_map_to_df(type_map, region_id=region_id)
    out_df = df_region.rename(columns={"Cell Type": "cell_type"})[["x", "y", "cell_type"]]
    out_path = os.path.join(SAVE_DIR, f"region_{region_id}.csv")
    out_df.to_csv(out_path, index=False)
    return out_path

for p in png_paths:
    try:
        out_path = process_one_image(p)
        written.append(out_path)
    except Exception as e:
        skipped.append((p, repr(e)))

print(f"✅ Wrote {len(written)} CSVs → {os.path.abspath(SAVE_DIR)}")
if skipped:
    print("⚠️ Skipped:")
    for fp, why in skipped:
        print(f"  - {fp}: {why}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

Original RGB min/max：
  R: 0.057373046875 1.0
  G: 0.062744140625 1.0
  B: 0.04052734375 1.0
valid pixel： 16233
Original RGB min/max：
  R: 0.041015625 1.0
  G: 0.061279296875 1.0
  B: 0.061767578125 1.0
valid pixel： 24469
Original RGB min/max：
  R: 0.06591796875 1.0
  G: 0.063720703125 1.0
  B: 0.049560546875 1.0
valid pixel： 9870
Original RGB min/max：
  R: 0.044677734375 1.0
  G: 0.06103515625 1.0
  B: 0.03857421875 1.0
valid pixel： 19608
Original RGB min/max：
  R: 0.05029296875 1.0
  G: 0.075927734375 1.0
  B: 0.05126953125 1.0
valid pixel： 14340
Original RGB min/max：
  R: 0.03564453125 1.0
  G: 0.0537109375 1.0
  B: 0.042724609375 1.0
valid pixel： 57473
Original RGB min/max：
  R: 0.04052734375 1.0
  G: 0.06494140625 1.0
  B: 0.04345703125 1.0
valid pixel： 30699
Original RGB min/max：
  R: 0.047119140625 1.0
  G: 0.06396484375 1.0
  B: 0.05322265625 1.0
valid pixel： 27392
Original RGB min/max：
  R: 0.036376953125 1.0
  G: 0.05810546875 1.0
  B: 0.045166015625 1.0
valid pixel： 27050
Or

In [ ]:
!mkdir -p /content/drive/MyDrive/decoded_regions_diffused_csv
!cp -r /content/decoded_regions_csv/* /content/drive/MyDrive/decoded_regions_diffused_csv/

